# Notebook 05: Classical Training and Tuning (OPTIMIZED)

**Purpose:** Train and tune classical ML models with research-backed optimization strategies

**Key Improvements:**
- Minimal SMOTE (4-6% synthetic data via undersample+SMOTE)
- Model-specific tuning methods (Optuna, HalvingGrid, Sequential)
- LinearSVC replaces SVC(linear) for speed
- Skip SVM RBF entirely (impractical for 42k samples)
- Separate cells per model (crash protection)
- Immediate saves after each tuning
- Class weighting for all models

**Models:** LinearSVC, Random Forest, XGBoost, Gradient Boosting, KNN, GMM

**Expected Time:** ~45-50 minutes

---

## 1. Import Libraries

In [2]:
import os, sys
from pathlib import Path
import json
from datetime import datetime
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight

# Models
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.mixture import GaussianMixture
from xgboost import XGBClassifier

# Imbalanced-learn
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# MLflow
import mlflow
import mlflow.sklearn

# Try to import advanced search methods
try:
    from sklearn.experimental import enable_halving_search_cv
    from sklearn.model_selection import HalvingGridSearchCV
    HALVING_AVAILABLE = True
    print('✓ HalvingGridSearchCV available')
except:
    HALVING_AVAILABLE = False
    print('⚠️  HalvingGridSearchCV not available (sklearn ≥1.2 required), using standard GridSearchCV')

try:
    import optuna
    from optuna.integration import OptunaSearchCV
    OPTUNA_AVAILABLE = True
    print('✓ Optuna available for Bayesian optimization')
    optuna.logging.set_verbosity(optuna.logging.WARNING)  # Reduce output
except:
    OPTUNA_AVAILABLE = False
    print('⚠️  Optuna not available, using GridSearchCV for XGBoost')

print('\n✓ All libraries imported successfully')

✓ HalvingGridSearchCV available
✓ Optuna available for Bayesian optimization

✓ All libraries imported successfully


## 2. Project Setup

In [3]:
# Paths
PROJECT_ROOT = Path.cwd().parent
FEATURES_DIR = PROJECT_ROOT / 'data' / 'features' / 'classical'
MODELS_DIR = PROJECT_ROOT / 'models' / 'classical'
RESULTS_DIR = PROJECT_ROOT / 'results'
FIGURES_DIR = RESULTS_DIR / 'figures'
LOGS_DIR = PROJECT_ROOT / 'logs'

# Create directories
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Features dir: {FEATURES_DIR}')
print(f'Models dir:   {MODELS_DIR}')
print(f'Results dir:  {RESULTS_DIR}')

Project root: /Users/harryirving/Development/projects/ai-ml/BikeAIv5
Features dir: /Users/harryirving/Development/projects/ai-ml/BikeAIv5/data/features/classical
Models dir:   /Users/harryirving/Development/projects/ai-ml/BikeAIv5/models/classical
Results dir:  /Users/harryirving/Development/projects/ai-ml/BikeAIv5/results


## 3. Setup MLflow

In [4]:
# MLflow configuration
mlflow.set_tracking_uri(f'file://{LOGS_DIR}/mlruns')
mlflow.set_experiment('angle_grinder_pipeline')

print(f'✓ MLflow tracking URI: {mlflow.get_tracking_uri()}')
print(f'✓ MLflow experiment: {mlflow.get_experiment_by_name("angle_grinder_pipeline").experiment_id}')

✓ MLflow tracking URI: file:///Users/harryirving/Development/projects/ai-ml/BikeAIv5/logs/mlruns
✓ MLflow experiment: 890722518077580718


## 4. Configuration - SELECT FEATURE SET

In [5]:
# ========================================
# CHANGE THIS TO SELECT FEATURE SET
# ========================================
FEATURE_SET = 'combined'  # Options: 'mfcc', 'gtcc', 'combined'
# ========================================

print('='*70)
print(f'FEATURE SET: {FEATURE_SET.upper()}')
print('='*70)

# Timing
notebook_start_time = time.time()

FEATURE SET: COMBINED


## 5. Load Features

In [6]:
# Load selected feature set
feature_file = FEATURES_DIR / f'{FEATURE_SET}_unbalanced_features.npy'

if not feature_file.exists():
    raise FileNotFoundError(f'Features not found: {feature_file}\nRun Notebook 03 first!')

labels_file = FEATURES_DIR / 'labels.npy'


print(f'Loading features from: {feature_file.name}')
data = np.load(feature_file)

X = np.load(feature_file)   # shape: (n_samples, n_features)
y = np.load(labels_file)    # shape: (n_samples,)

print(f'\n✓ Loaded successfully')
print(f'  Features shape: {X.shape}')
print(f'  Labels shape:   {y.shape}')
print(f'  Feature count:  {X.shape[1]}')
print(f'  Sample count:   {X.shape[0]:,}')

# Class distribution
unique, counts = np.unique(y, return_counts=True)
total = len(y)
print(f'\nClass distribution:')
for label, count in zip(unique, counts):
    class_name = 'Grinder' if label == 1 else 'Non-grinder'
    print(f'  {class_name} (class {label}): {count:,} ({count/total*100:.1f}%)')

Loading features from: combined_unbalanced_features.npy

✓ Loaded successfully
  Features shape: (59869, 248)
  Labels shape:   (59869,)
  Feature count:  248
  Sample count:   59,869

Class distribution:
  Non-grinder (class 0): 27,077 (45.2%)
  Grinder (class 1): 32,792 (54.8%)


## 6. Check and Handle NaN Values

In [7]:
print('Checking for NaN values...')
nan_count = np.isnan(X).sum()

if nan_count > 0:
    print(f'⚠️  Found {nan_count:,} NaN values ({nan_count/(X.shape[0]*X.shape[1])*100:.3f}% of data)')
    print('   Applying median imputation...')
    
    imputer = SimpleImputer(strategy='median')
    X = imputer.fit_transform(X)
    
    # Verify
    remaining_nans = np.isnan(X).sum()
    if remaining_nans == 0:
        print('   ✓ All NaN values handled')
    else:
        print(f'   ⚠️  Still have {remaining_nans} NaN values')
else:
    print('✓ No NaN values detected')

Checking for NaN values...
✓ No NaN values detected


## 7. Train/Val/Test Split

In [8]:
print('Splitting  70% train / 15% val / 15% test')

# First split: separate test set (15%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, 
    test_size=0.15, 
    random_state=42, 
    stratify=y
)

# Second split: separate val from train (15% of remaining = ~17.6% of temp)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.176,  # 0.15 / 0.85 ≈ 0.176
    random_state=42,
    stratify=y_temp
)

print(f'\n✓ Split complete')
print(f'  Train: {X_train.shape[0]:,} samples')
print(f'  Val:   {X_val.shape[0]:,} samples')
print(f'  Test:  {X_test.shape[0]:,} samples')

# Show class distribution in each set
for name, labels in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    unique, counts = np.unique(labels, return_counts=True)
    print(f'\n  {name} distribution:')
    for label, count in zip(unique, counts):
        class_name = 'Grinder' if label == 1 else 'Non-grinder'
        print(f'    {class_name}: {count:,} ({count/len(labels)*100:.1f}%)')

Splitting  70% train / 15% val / 15% test

✓ Split complete
  Train: 41,931 samples
  Val:   8,957 samples
  Test:  8,981 samples

  Train distribution:
    Non-grinder: 18,964 (45.2%)
    Grinder: 22,967 (54.8%)

  Val distribution:
    Non-grinder: 4,051 (45.2%)
    Grinder: 4,906 (54.8%)

  Test distribution:
    Non-grinder: 4,062 (45.2%)
    Grinder: 4,919 (54.8%)


## 8. Optimal Balancing: Minimal SMOTE Strategy

In [ ]:
print('Applying OPTIMAL BALANCING: Undersample + Minimal SMOTE')
print('='*70)

# Current distribution
unique_train, counts_train = np.unique(y_train, return_counts=True)
class_counts = dict(zip(unique_train, counts_train))

grinder_count = class_counts.get(1, 0)
non_grinder_count = class_counts.get(0, 0)
total_train = len(y_train)

print(f'\nBEFORE balancing:')
print(f'  Grinder:     {grinder_count:,} ({grinder_count/total_train*100:.1f}%)')
print(f'  Non-grinder: {non_grinder_count:,} ({non_grinder_count/total_train*100:.1f}%)')
print(f'  Total:       {total_train:,}')

# Target: Meet in the middle for perfect 1:1 with minimal synthetic
TARGET_PER_CLASS = int((grinder_count + non_grinder_count) / 2)

print(f'\nTarget per class: {TARGET_PER_CLASS:,} (perfect 1:1 balance)')

# Step 1: Undersample majority class
print('\nStep 1: Undersampling majority class...')
majority_class = 1 if grinder_count > non_grinder_count else 0
minority_class = 0 if majority_class == 1 else 1

rus = RandomUnderSampler(
    sampling_strategy={majority_class: TARGET_PER_CLASS},
    random_state=42
)
X_train_under, y_train_under = rus.fit_resample(X_train, y_train)

removed_samples = (grinder_count if majority_class == 1 else non_grinder_count) - TARGET_PER_CLASS
print(f'  Removed {removed_samples:,} samples from majority class')

# Step 2: SMOTE minority class
print('\nStep 2: SMOTE minority class to match...')
smote = SMOTE(
    sampling_strategy={minority_class: TARGET_PER_CLASS},
    random_state=42
)
X_train_balanced, y_train = smote.fit_resample(X_train_under, y_train_under)

added_samples = TARGET_PER_CLASS - (non_grinder_count if minority_class == 0 else grinder_count)
print(f'  Added {added_samples:,} synthetic samples to minority class')

# Final statistics
unique_balanced, counts_balanced = np.unique(y_train, return_counts=True)
total_balanced = len(y_train)
synthetic_percentage = (added_samples / total_balanced) * 100

print(f'\nAFTER balancing:')
print(f'  Grinder:     {TARGET_PER_CLASS:,} (50.0%)')
print(f'  Non-grinder: {TARGET_PER_CLASS:,} (50.0%)')
print(f'  Total:       {total_balanced:,}')
print(f'\n  Real       {total_balanced - added_samples:,} ({(1-synthetic_percentage/100)*100:.1f}%)')
print(f'  Synthetic  {added_samples:,} ({synthetic_percentage:.1f}%) ← Minimal!')
print(f'\n✓ Optimal balancing complete: {synthetic_percentage:.1f}% synthetic vs 40-50% typical')
print('='*70)

## 9. Feature Scaling

In [10]:
print('Applying StandardScaler...')

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f'✓ Scaling complete')
print(f'  Train: {X_train_scaled.shape}')
print(f'  Val:   {X_val_scaled.shape}')
print(f'  Test:  {X_test_scaled.shape}')

# Save scaler
scaler_path = MODELS_DIR / f'scaler_{FEATURE_SET}.pkl'
joblib.dump(scaler, scaler_path)
print(f'\n✓ Scaler saved: {scaler_path.name}')

Applying StandardScaler...
✓ Scaling complete
  Train: (41931, 248)
  Val:   (8957, 248)
  Test:  (8981, 248)

✓ Scaler saved: scaler_combined.pkl


## 10. Define Baseline Models (WITH OPTIMIZATIONS)

In [17]:
print('Defining baseline models with class weighting...')
print('='*70)

# Calculate scale_pos_weight for XGBoost
n_grinder = (y_train == 1).sum()
n_non_grinder = (y_train == 0).sum()
scale_pos_weight = n_non_grinder / n_grinder
print(f'XGBoost scale_pos_weight: {scale_pos_weight:.2f}\n')

# LinearSVC (wrapped for probability estimates)
linear_svc_base = LinearSVC(
    random_state=42,
    max_iter=5000,
    dual=False,  # Faster when n_samples > n_features
    loss='squared_hinge',
    tol=1e-4,
    class_weight='balanced'
)

baseline_models = {
    'LinearSVC': CalibratedClassifierCV(linear_svc_base, cv=3),
    
    'Random Forest': RandomForestClassifier(
        random_state=42,
        n_estimators=100,
        class_weight='balanced',
        max_features='sqrt',
        min_samples_leaf=2,
        n_jobs=-1
    ),
    
    'XGBoost': XGBClassifier(
        random_state=42,
        eval_metric='logloss',
        scale_pos_weight=scale_pos_weight,
        tree_method='hist',
        n_jobs=-1
    ),
    
    'Gradient Boosting': GradientBoostingClassifier(
        random_state=42,
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        subsample=0.8,
        min_samples_leaf=2
        
    ),
    
    'KNN': KNeighborsClassifier(
        n_neighbors=5,
        weights='distance',
        metric='euclidean',
        n_jobs=-1
    )
}

print(f'Defined {len(baseline_models)} baseline models:')
for name in baseline_models.keys():
    print(f'  ✓ {name}')
print('  ✓ GMM (trained separately)')
print('\nNOTE: SVM RBF skipped (impractical for 42k samples - would take 90+ min)')
print('='*70)

Defining baseline models with class weighting...
XGBoost scale_pos_weight: 0.83

Defined 5 baseline models:
  ✓ LinearSVC
  ✓ Random Forest
  ✓ XGBoost
  ✓ Gradient Boosting
  ✓ KNN
  ✓ GMM (trained separately)

NOTE: SVM RBF skipped (impractical for 42k samples - would take 90+ min)


## 11. Train Baseline Models

In [18]:
print('\nTraining Baseline Models...')
print('='*70)

baseline_results = []
baseline_start_time = time.time()

# Calculate sample weights for Gradient Boosting (no class_weight parameter)
sample_weights = compute_sample_weight('balanced', y_train)

for name, model in baseline_models.items():
    print(f'\n{name}...')
    start = time.time()
    
    # Special handling for Gradient Boosting
    if name == 'Gradient Boosting':
        model.fit(X_train_scaled, y_train, sample_weight=sample_weights)
    else:
        model.fit(X_train_scaled, y_train)
    
    train_time = time.time() - start
    
    # Predictions
    y_train_pred = model.predict(X_train_scaled)
    y_val_pred = model.predict(X_val_scaled)
    y_test_pred = model.predict(X_test_scaled)
    
    # Metrics
    train_acc = accuracy_score(y_train, y_train_pred)
    val_acc = accuracy_score(y_val, y_val_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    val_f1 = f1_score(y_val, y_val_pred)
    test_f1 = f1_score(y_test, y_test_pred)
    
    # Cross-validation (reduced to 3 folds for speed)
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=3, scoring='f1', n_jobs=-1)
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()
    
    # Store results
    baseline_results.append({
        'Model': name,
        'Train Acc': train_acc,
        'Val Acc': val_acc,
        'Test Acc': test_acc,
        'Val F1': val_f1,
        'Test F1': test_f1,
        'CV Mean': cv_mean,
        'CV Std': cv_std,
        'Train Time (s)': train_time
    })
    
    # Print results
    print(f'  Train Acc: {train_acc:.4f}')
    print(f'  Val Acc:   {val_acc:.4f}')
    print(f'  Test Acc:  {test_acc:.4f}')
    print(f'  Val F1:    {val_f1:.4f}')
    print(f'  Test F1:   {test_f1:.4f}')
    print(f'  CV:        {cv_mean:.4f} ± {cv_std:.4f}')
    print(f'  Time:      {train_time:.1f}s')

baseline_total_time = time.time() - baseline_start_time

# Create DataFrame
df_baseline = pd.DataFrame(baseline_results)

print('='*70)
print('BASELINE RESULTS:')
print('='*70)
print(df_baseline[['Model', 'Test Acc', 'Test F1', 'CV Mean', 'Train Time (s)']].to_string(index=False))
print(f'\nTotal baseline training time: {baseline_total_time/60:.1f} minutes')
print('='*70)

# Save baseline results
baseline_csv = RESULTS_DIR / f'05_baseline_results_{FEATURE_SET}.csv'
df_baseline.to_csv(baseline_csv, index=False)
print(f'\n✓ Saved: {baseline_csv.name}')


Training Baseline Models...

LinearSVC...
  Train Acc: 0.9446
  Val Acc:   0.9403
  Test Acc:  0.9419
  Val F1:    0.9462
  Test F1:   0.9476
  CV:        0.9473 ± 0.0017
  Time:      11.8s

Random Forest...
  Train Acc: 0.9995
  Val Acc:   0.9753
  Test Acc:  0.9774
  Val F1:    0.9778
  Test F1:   0.9796
  CV:        0.9766 ± 0.0010
  Time:      8.9s

XGBoost...
  Train Acc: 1.0000
  Val Acc:   0.9824
  Test Acc:  0.9853
  Val F1:    0.9840
  Test F1:   0.9867
  CV:        0.9841 ± 0.0005
  Time:      1.3s

Gradient Boosting...
  Train Acc: 0.9855
  Val Acc:   0.9744
  Test Acc:  0.9758
  Val F1:    0.9768
  Test F1:   0.9781
  CV:        0.9749 ± 0.0008
  Time:      299.7s

KNN...
  Train Acc: 1.0000
  Val Acc:   0.9709
  Test Acc:  0.9708
  Val F1:    0.9738
  Test F1:   0.9737
  CV:        0.9694 ± 0.0005
  Time:      0.0s
BASELINE RESULTS:
            Model  Test Acc  Test F1  CV Mean  Train Time (s)
        LinearSVC  0.941877 0.947590 0.947277       11.843783
    Random Forest

## 12. Train GMM (Baseline)

In [19]:
print('\nTraining GMM (Gaussian Mixture Model)...')
print('='*70)

gmm_start = time.time()

# Train GMM with baseline n_components
gmm_baseline = GaussianMixture(
    n_components=6,
    covariance_type='diag',
    random_state=42,
    max_iter=100
)

gmm_baseline.fit(X_train_scaled)

# Predict clusters on training data
train_clusters = gmm_baseline.predict(X_train_scaled)

# Map clusters to classes based on training data majority vote
cluster_to_class = {}
print('\nCluster to class mapping:')
for cluster_id in range(6):
    mask = train_clusters == cluster_id
    if mask.sum() > 0:
        cluster_labels = y_train[mask]
        majority_class = np.bincount(cluster_labels).argmax()
        cluster_to_class[cluster_id] = majority_class
        
        n_grinder = (cluster_labels == 1).sum()
        n_non_grinder = (cluster_labels == 0).sum()
        total = len(cluster_labels)
        print(f'  Cluster {cluster_id} → Class {majority_class} ({"Grinder" if majority_class == 1 else "Non-grinder"})')
        print(f'    Composition: {n_grinder} grinder, {n_non_grinder} non-grinder (n={total})')

# Function to predict using cluster mapping
def gmm_predict(X):
    clusters = gmm_baseline.predict(X)
    return np.array([cluster_to_class[c] for c in clusters])

# Evaluate
y_train_pred_gmm = gmm_predict(X_train_scaled)
y_val_pred_gmm = gmm_predict(X_val_scaled)
y_test_pred_gmm = gmm_predict(X_test_scaled)

train_acc_gmm = accuracy_score(y_train, y_train_pred_gmm)
val_acc_gmm = accuracy_score(y_val, y_val_pred_gmm)
test_acc_gmm = accuracy_score(y_test, y_test_pred_gmm)
val_f1_gmm = f1_score(y_val, y_val_pred_gmm)
test_f1_gmm = f1_score(y_test, y_test_pred_gmm)

gmm_time = time.time() - gmm_start

print(f'\nGMM Baseline Performance:')
print(f'  Train Acc: {train_acc_gmm:.4f}')
print(f'  Val Acc:   {val_acc_gmm:.4f}')
print(f'  Test Acc:  {test_acc_gmm:.4f}')
print(f'  Val F1:    {val_f1_gmm:.4f}')
print(f'  Test F1:   {test_f1_gmm:.4f}')
print(f'  Time:      {gmm_time:.1f}s')

# Add to baseline results
baseline_results.append({
    'Model': 'GMM',
    'Train Acc': train_acc_gmm,
    'Val Acc': val_acc_gmm,
    'Test Acc': test_acc_gmm,
    'Val F1': val_f1_gmm,
    'Test F1': test_f1_gmm,
    'CV Mean': np.nan,  # GMM doesn't use CV
    'CV Std': np.nan,
    'Train Time (s)': gmm_time
})

# Update DataFrame
df_baseline = pd.DataFrame(baseline_results)
df_baseline.to_csv(baseline_csv, index=False)

print('\n✓ GMM baseline complete')
print('='*70)


Training GMM (Gaussian Mixture Model)...

Cluster to class mapping:
  Cluster 0 → Class 1 (Grinder)
    Composition: 1342 grinder, 786 non-grinder (n=2128)
  Cluster 1 → Class 0 (Non-grinder)
    Composition: 3039 grinder, 6854 non-grinder (n=9893)
  Cluster 2 → Class 1 (Grinder)
    Composition: 4282 grinder, 1515 non-grinder (n=5797)
  Cluster 3 → Class 0 (Non-grinder)
    Composition: 770 grinder, 3143 non-grinder (n=3913)
  Cluster 4 → Class 1 (Grinder)
    Composition: 3229 grinder, 1961 non-grinder (n=5190)
  Cluster 5 → Class 1 (Grinder)
    Composition: 10305 grinder, 4705 non-grinder (n=15010)

GMM Baseline Performance:
  Train Acc: 0.6953
  Val Acc:   0.6837
  Test Acc:  0.6924
  Val F1:    0.7404
  Test F1:   0.7475
  Time:      2.3s

✓ GMM baseline complete


In [20]:
print('\n' + '='*70)
print('DIAGNOSTIC: Testing on ORIGINAL imbalanced data')
print('='*70)

# Train XGBoost on ORIGINAL imbalanced data (no SMOTE) for comparison
print('\nTraining XGBoost on RAW imbalanced data...')

xgb_raw = XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    scale_pos_weight=n_non_grinder / n_grinder,
    tree_method='hist',
    n_jobs=-1
)

# Use ORIGINAL X_train (before SMOTE), scaled
X_train_original_scaled = scaler.transform(X_train)  # Before balancing
xgb_raw.fit(X_train_original_scaled, y_train)

# Test on validation
y_val_pred_raw = xgb_raw.predict(X_val_scaled)
val_f1_raw = f1_score(y_val, y_val_pred_raw)
val_acc_raw = accuracy_score(y_val, y_val_pred_raw)

print(f'\nPerformance on RAW imbalanced ')
print(f'  Val F1:  {val_f1_raw:.4f}')
print(f'  Val Acc: {val_acc_raw:.4f}')

# Compare with baseline on SMOTE data - USE df_baseline not baseline_results
xgb_baseline_row = df_baseline[df_baseline['Model'] == 'XGBoost']

if len(xgb_baseline_row) > 0:
    xgb_baseline_val_f1 = xgb_baseline_row['Val F1'].values[0]
    xgb_baseline_test_f1 = xgb_baseline_row['Test F1'].values[0]
    
    print(f'\nComparison:')
    print(f'  SMOTE baseline Val F1:  {xgb_baseline_val_f1:.4f}')
    print(f'  RAW data Val F1:        {val_f1_raw:.4f}')
    print(f'  Difference:             {xgb_baseline_val_f1 - val_f1_raw:+.4f}')
    
    if abs(xgb_baseline_val_f1 - val_f1_raw) < 0.02:
        print('\n⚠️  WARNING: SMOTE providing minimal benefit (<2% improvement)!')
        print('   Consider training on raw data with class_weight instead')
    elif val_f1_raw > xgb_baseline_val_f1:
        print('\n⚠️  CRITICAL: RAW data performs BETTER than SMOTE!')
        print('   SMOTE is hurting performance - skip balancing!')
    else:
        print(f'\n✓ SMOTE is helping (+{xgb_baseline_val_f1 - val_f1_raw:.2%} improvement)')
else:
    print('\n⚠️  XGBoost baseline not found in df_baseline')

# Also show baseline results table for context
print('\n' + '-'*70)
print('BASELINE RESULTS (all models):')
print('-'*70)
print(df_baseline[['Model', 'Val F1', 'Test F1', 'Val Acc', 'Test Acc']].to_string(index=False))

print('='*70)




DIAGNOSTIC: Testing on ORIGINAL imbalanced data

Training XGBoost on RAW imbalanced data...

Performance on RAW imbalanced 
  Val F1:  0.9850
  Val Acc: 0.9835

Comparison:
  SMOTE baseline Val F1:  0.9840
  RAW data Val F1:        0.9850
  Difference:             -0.0010

⚠️  WARNING: SMOTE providing minimal benefit (<2% improvement)!
   Consider training on raw data with class_weight instead

----------------------------------------------------------------------
BASELINE RESULTS (all models):
----------------------------------------------------------------------
            Model   Val F1  Test F1  Val Acc  Test Acc
        LinearSVC 0.946237 0.947590 0.940270  0.941877
    Random Forest 0.977818 0.979625 0.975327  0.977397
          XGBoost 0.984015 0.986683 0.982360  0.985302
Gradient Boosting 0.976834 0.978114 0.974433  0.975838
              KNN 0.973761 0.973726 0.970861  0.970827
              GMM 0.740354 0.747510 0.683711  0.692351


## 13. Hyperparameter Tuning Setup

In [21]:
print('\nHYPERPARAMETER TUNING SETUP')
print('='*70)

# Initialize storage
tuned_models = {}
tuning_results = []

# Models to tune
MODELS_TO_TUNE = ['XGBoost', 'Random Forest', 'Gradient Boosting', 'KNN', 'LinearSVC']

print(f'Models to tune: {len(MODELS_TO_TUNE)}')
for model_name in MODELS_TO_TUNE:
    print(f'  ✓ {model_name}')

print(f'\nSkipping: GMM (will optimize separately with BIC)')
print(f'\nEstimated total tuning time: ~35-40 minutes')
print('\nIMPORTANT: Each model is in a SEPARATE CELL below')
print('           If one crashes, others are safe!')
print('           Models are saved immediately after tuning.')
print('='*70)


HYPERPARAMETER TUNING SETUP
Models to tune: 5
  ✓ XGBoost
  ✓ Random Forest
  ✓ Gradient Boosting
  ✓ KNN
  ✓ LinearSVC

Skipping: GMM (will optimize separately with BIC)

Estimated total tuning time: ~35-40 minutes

IMPORTANT: Each model is in a SEPARATE CELL below
           If one crashes, others are safe!
           Models are saved immediately after tuning.


---
# HYPERPARAMETER TUNING (One Cell Per Model)
**Run each cell independently. Models are saved immediately after tuning.**

---

## 14A. Tune XGBoost (PRIMARY MODEL)

In [23]:
print('\n' + '='*70)
print('TUNING: XGBoost (Bayesian Optimization)')
print('='*70)

model_name = 'XGBoost'
start_time = time.time()

if OPTUNA_AVAILABLE:
    print('Using Optuna for Bayesian optimization...\n')
    
    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=50),
            'max_depth': trial.suggest_int('max_depth', 3, 8),
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
            'subsample': trial.suggest_float('subsample', 0.7, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'gamma': trial.suggest_float('gamma', 0, 0.5),
            'random_state': 42,
            'eval_metric': 'logloss',
            'scale_pos_weight': scale_pos_weight,
            'tree_method': 'hist',
            'n_jobs': -1
        }
        
        model = XGBClassifier(**params)
        scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='f1', n_jobs=-1)
        return scores.mean()
    
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=50, show_progress_bar=True)
    
    print(f'\n✓ Optuna optimization complete')
    print(f'  Best trial: {study.best_trial.number}')
    print(f'  Best CV F1: {study.best_value:.4f}')
    print(f'  Best parameters:')
    for key, value in study.best_params.items():
        print(f'    {key}: {value}')
    
    # Train final model with best params
    best_params = study.best_params
    best_params.update({
        'random_state': 42,
        'eval_metric': 'logloss',
        'scale_pos_weight': scale_pos_weight,
        'tree_method': 'hist',
        'n_jobs': -1
    })
    best_model = XGBClassifier(**best_params)
    best_model.fit(X_train_scaled, y_train)
    
else:
    print('Using HalvingGridSearchCV (Optuna not available)...\n')
    
    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.05, 0.1],
        'subsample': [0.7, 0.8, 0.9],
        'colsample_bytree': [0.7, 0.8, 0.9],
        'min_child_weight': [1, 3, 5]
    }
    
    base_model = XGBClassifier(
        random_state=42,
        eval_metric='logloss',
        scale_pos_weight=scale_pos_weight,
        tree_method='hist',
        n_jobs=-1
    )
    
    if HALVING_AVAILABLE:
        search = HalvingGridSearchCV(
            base_model,
            param_grid,
            cv=5,
            scoring='f1',
            factor=3,
            resource='n_samples',
            max_resources=len(X_train_scaled),
            n_jobs=-1,
            verbose=2
        )
    else:
        search = GridSearchCV(
            base_model,
            param_grid,
            cv=5,
            scoring='f1',
            n_jobs=-1,
            verbose=2
        )
    
    search.fit(X_train_scaled, y_train)
    best_model = search.best_estimator_
    print(f'\n✓ Grid search complete')
    print(f'  Best CV F1: {search.best_score_:.4f}')
    print(f'  Best parameters: {search.best_params_}')

# Evaluate
y_train_pred = best_model.predict(X_train_scaled)
y_val_pred = best_model.predict(X_val_scaled)
y_test_pred = best_model.predict(X_test_scaled)

train_acc = accuracy_score(y_train, y_train_pred)
val_acc = accuracy_score(y_val, y_val_pred)
test_acc = accuracy_score(y_test, y_test_pred)
val_f1 = f1_score(y_val, y_val_pred)
test_f1 = f1_score(y_test, y_test_pred)

tune_time = time.time() - start_time

# Store
tuned_models[model_name] = best_model
tuning_results.append({
    'Model': model_name,
    'Train Acc': train_acc,
    'Val Acc': val_acc,
    'Test Acc': test_acc,
    'Val F1': val_f1,
    'Test F1': test_f1,
    'Tune Time (s)': tune_time
})

# Save immediately
model_path = MODELS_DIR / f'xgboost_tuned_{FEATURE_SET}.pkl'
joblib.dump(best_model, model_path)

print(f'\n✓ {model_name} COMPLETE')
print(f'  Test F1:  {test_f1:.4f}')
print(f'  Test Acc: {test_acc:.4f}')
print(f'  Time:     {tune_time/60:.1f} minutes')
print(f'  Saved:    {model_path.name}')
print('='*70)


TUNING: XGBoost (Bayesian Optimization)
Using Optuna for Bayesian optimization...



  0%|          | 0/50 [00:00<?, ?it/s]


✓ Optuna optimization complete
  Best trial: 49
  Best CV F1: 0.9875
  Best parameters:
    n_estimators: 500
    max_depth: 8
    learning_rate: 0.16409441656310955
    subsample: 0.9384973636402726
    colsample_bytree: 0.9389072223958721
    min_child_weight: 2
    gamma: 0.026778723409013012

✓ XGBoost COMPLETE
  Test F1:  0.9886
  Test Acc: 0.9874
  Time:     12.0 minutes
  Saved:    xgboost_tuned_combined.pkl


## 14B. Tune Random Forest

In [24]:
print('\n' + '='*70)
print('TUNING: Random Forest (HalvingGrid or Sequential)')
print('='*70)

model_name = 'Random Forest'
start_time = time.time()

if HALVING_AVAILABLE:
    print('Using HalvingGridSearchCV...\n')
    
    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [10, 15, 20, 25],
        'min_samples_split': [5, 10, 15],
        'min_samples_leaf': [2, 4, 6],
        'max_features': ['sqrt', 'log2', 0.5]
    }
    
    base_model = RandomForestClassifier(
        random_state=42,
        class_weight='balanced',
        n_jobs=-1
    )
    
    search = HalvingGridSearchCV(
        base_model,
        param_grid,
        cv=5,
        scoring='f1',
        factor=3,
        resource='n_samples',
        max_resources=len(X_train_scaled),
        n_jobs=-1,
        verbose=2
    )
    
    search.fit(X_train_scaled, y_train)
    best_model = search.best_estimator_
    
    print(f'\n✓ HalvingGridSearchCV complete')
    print(f'  Best CV F1: {search.best_score_:.4f}')
    print(f'  Best parameters: {search.best_params_}')
    
else:
    print('Using Sequential Coarse→Fine GridSearchCV...\n')
    
    # Stage 1: Coarse
    print('Stage 1: Coarse search...')
    param_coarse = {
        'n_estimators': [100, 300],
        'max_depth': [10, 20],
        'min_samples_split': [5, 10]
    }
    
    base_model = RandomForestClassifier(
        random_state=42,
        class_weight='balanced',
        n_jobs=-1
    )
    
    grid_coarse = GridSearchCV(base_model, param_coarse, cv=5, scoring='f1', n_jobs=-1, verbose=1)
    grid_coarse.fit(X_train_scaled, y_train)
    
    best_coarse = grid_coarse.best_params_
    print(f'  Coarse best: {best_coarse}')
    
    # Stage 2: Fine
    print('\nStage 2: Fine search around best...')
    param_fine = {
        'n_estimators': [max(100, best_coarse['n_estimators']-50), best_coarse['n_estimators'], best_coarse['n_estimators']+50],
        'max_depth': [max(5, best_coarse['max_depth']-5), best_coarse['max_depth'], best_coarse['max_depth']+5],
        'min_samples_split': [best_coarse['min_samples_split']],
        'min_samples_leaf': [2, 4, 6],
        'max_features': ['sqrt', 'log2']
    }
    
    grid_fine = GridSearchCV(base_model, param_fine, cv=5, scoring='f1', n_jobs=-1, verbose=1)
    grid_fine.fit(X_train_scaled, y_train)
    
    best_model = grid_fine.best_estimator_
    
    print(f'\n✓ Sequential search complete')
    print(f'  Best CV F1: {grid_fine.best_score_:.4f}')
    print(f'  Best parameters: {grid_fine.best_params_}')

# Evaluate
y_train_pred = best_model.predict(X_train_scaled)
y_val_pred = best_model.predict(X_val_scaled)
y_test_pred = best_model.predict(X_test_scaled)

train_acc = accuracy_score(y_train, y_train_pred)
val_acc = accuracy_score(y_val, y_val_pred)
test_acc = accuracy_score(y_test, y_test_pred)
val_f1 = f1_score(y_val, y_val_pred)
test_f1 = f1_score(y_test, y_test_pred)

tune_time = time.time() - start_time

# Store
tuned_models[model_name] = best_model
tuning_results.append({
    'Model': model_name,
    'Train Acc': train_acc,
    'Val Acc': val_acc,
    'Test Acc': test_acc,
    'Val F1': val_f1,
    'Test F1': test_f1,
    'Tune Time (s)': tune_time
})

# Save immediately
model_path = MODELS_DIR / f'random_forest_tuned_{FEATURE_SET}.pkl'
joblib.dump(best_model, model_path)

print(f'\n✓ {model_name} COMPLETE')
print(f'  Test F1:  {test_f1:.4f}')
print(f'  Test Acc: {test_acc:.4f}')
print(f'  Time:     {tune_time/60:.1f} minutes')
print(f'  Saved:    {model_path.name}')
print('='*70)


TUNING: Random Forest (HalvingGrid or Sequential)
Using HalvingGridSearchCV...

n_iterations: 5
n_required_iterations: 6
n_possible_iterations: 5
min_resources_: 172
max_resources_: 41931
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 324
n_resources: 172
Fitting 5 folds for each of 324 candidates, totalling 1620 fits
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=2, min_samples_split=5, n_estimators=100; total time=   0.1s
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=2, min_samples_split=5, n_estimators=100; total time=   0.1s
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=2, min_samples_split=5, n_estimators=100; total time=   0.1s
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=2, min_samples_split=5, n_estimators=100; total time=   0.1s
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=2, min_samples_split=5, n_estimators=100; total time=   0.2s
[CV] END max_depth=10, max_features=sqrt, min_samples

## 14C. Tune Gradient Boosting

In [25]:
print('\n' + '='*70)
print('TUNING: Gradient Boosting (Simplified Single-Stage)')
print('='*70)

model_name = 'Gradient Boosting'
start_time = time.time()

# Recalculate sample weights
sample_weights = compute_sample_weight('balanced', y_train)

# SIMPLIFIED: One stage with reduced grid
print('\nOptimizing with simplified parameter grid...')
param_grid = {
    'learning_rate': [0.05, 0.1, 0.15],
    'max_depth': [3, 5, 7],
    'n_estimators': [100, 200, 300],
    'subsample': [0.8, 0.9],
    'min_samples_split': [5, 10],
    'min_samples_leaf': [2, 4]
}

model_gb = GradientBoostingClassifier(random_state=42)

# Use 3-fold CV instead of 5 for speed
grid_search = GridSearchCV(
    model_gb, 
    param_grid, 
    cv=3,  # Reduced from 5
    scoring='f1', 
    n_jobs=-1, 
    verbose=2
)

# Note: GridSearchCV doesn't pass sample_weight, so we accept this limitation
# for speed. The class imbalance is already handled by SMOTE.
grid_search.fit(X_train_scaled, y_train)

best_model = grid_search.best_estimator_

print(f'\n✓ Grid search complete')
print(f'  Best CV F1: {grid_search.best_score_:.4f}')
print(f'  Best parameters:')
for key, val in grid_search.best_params_.items():
    print(f'    {key}: {val}')

# Evaluate
y_train_pred = best_model.predict(X_train_scaled)
y_val_pred = best_model.predict(X_val_scaled)
y_test_pred = best_model.predict(X_test_scaled)

train_acc = accuracy_score(y_train, y_train_pred)
val_acc = accuracy_score(y_val, y_val_pred)
test_acc = accuracy_score(y_test, y_test_pred)
val_f1 = f1_score(y_val, y_val_pred)
test_f1 = f1_score(y_test, y_test_pred)

tune_time = time.time() - start_time

# Store
tuned_models[model_name] = best_model
tuning_results.append({
    'Model': model_name,
    'Train Acc': train_acc,
    'Val Acc': val_acc,
    'Test Acc': test_acc,
    'Val F1': val_f1,
    'Test F1': test_f1,
    'Tune Time (s)': tune_time
})

# Save immediately
model_path = MODELS_DIR / f'gradient_boosting_tuned_{FEATURE_SET}.pkl'
joblib.dump(best_model, model_path)

print(f'\n✓ {model_name} COMPLETE')
print(f'  Test F1:  {test_f1:.4f}')
print(f'  Test Acc: {test_acc:.4f}')
print(f'  Time:     {tune_time/60:.1f} minutes (simplified tuning)')
print(f'  Saved:    {model_path.name}')
print('='*70)




TUNING: Gradient Boosting (Simplified Single-Stage)

Optimizing with simplified parameter grid...
Fitting 3 folds for each of 216 candidates, totalling 648 fits
[CV] END learning_rate=0.05, max_depth=3, min_samples_leaf=2, min_samples_split=5, n_estimators=100, subsample=0.8; total time= 4.0min
[CV] END learning_rate=0.05, max_depth=3, min_samples_leaf=2, min_samples_split=5, n_estimators=100, subsample=0.8; total time= 4.0min
[CV] END learning_rate=0.05, max_depth=3, min_samples_leaf=2, min_samples_split=5, n_estimators=100, subsample=0.8; total time= 4.0min
[CV] END learning_rate=0.05, max_depth=3, min_samples_leaf=2, min_samples_split=5, n_estimators=100, subsample=0.9; total time= 4.5min
[CV] END learning_rate=0.05, max_depth=3, min_samples_leaf=2, min_samples_split=5, n_estimators=100, subsample=0.9; total time= 4.5min
[CV] END learning_rate=0.05, max_depth=3, min_samples_leaf=2, min_samples_split=5, n_estimators=100, subsample=0.9; total time= 4.6min


KeyboardInterrupt: 

## 14D. Tune KNN

In [26]:
print('\n' + '='*70)
print('TUNING: KNN (Exhaustive GridSearchCV)')
print('='*70)

model_name = 'KNN'
start_time = time.time()

print('Using GridSearchCV (small search space)...\n')

param_grid = {
    'n_neighbors': [5, 7, 9, 11, 13, 15],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski'],
    'p': [1, 2]
}

base_model = KNeighborsClassifier(n_jobs=-1)

grid_search = GridSearchCV(
    base_model,
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train_scaled, y_train)
best_model = grid_search.best_estimator_

print(f'\n✓ Grid search complete')
print(f'  Best CV F1: {grid_search.best_score_:.4f}')
print(f'  Best parameters: {grid_search.best_params_}')

# Evaluate
y_train_pred = best_model.predict(X_train_scaled)
y_val_pred = best_model.predict(X_val_scaled)
y_test_pred = best_model.predict(X_test_scaled)

train_acc = accuracy_score(y_train, y_train_pred)
val_acc = accuracy_score(y_val, y_val_pred)
test_acc = accuracy_score(y_test, y_test_pred)
val_f1 = f1_score(y_val, y_val_pred)
test_f1 = f1_score(y_test, y_test_pred)

tune_time = time.time() - start_time

# Store
tuned_models[model_name] = best_model
tuning_results.append({
    'Model': model_name,
    'Train Acc': train_acc,
    'Val Acc': val_acc,
    'Test Acc': test_acc,
    'Val F1': val_f1,
    'Test F1': test_f1,
    'Tune Time (s)': tune_time
})

# Save immediately
model_path = MODELS_DIR / f'knn_tuned_{FEATURE_SET}.pkl'
joblib.dump(best_model, model_path)

print(f'\n✓ {model_name} COMPLETE')
print(f'  Test F1:  {test_f1:.4f}')
print(f'  Test Acc: {test_acc:.4f}')
print(f'  Time:     {tune_time/60:.1f} minutes')
print(f'  Saved:    {model_path.name}')
print(f'\nNOTE: KNN for comparison only - not production-ready (slow inference)')
print('='*70)


TUNING: KNN (Exhaustive GridSearchCV)
Using GridSearchCV (small search space)...

Fitting 5 folds for each of 72 candidates, totalling 360 fits
[CV] END metric=euclidean, n_neighbors=5, p=1, weights=distance; total time=   6.5s
[CV] END metric=euclidean, n_neighbors=5, p=1, weights=distance; total time=   6.5s
[CV] END metric=euclidean, n_neighbors=5, p=1, weights=distance; total time=   6.6s
[CV] END metric=euclidean, n_neighbors=5, p=1, weights=uniform; total time=   6.6s
[CV] END metric=euclidean, n_neighbors=5, p=1, weights=distance; total time=   6.6s
[CV] END metric=euclidean, n_neighbors=5, p=1, weights=uniform; total time=   6.7s
[CV] END metric=euclidean, n_neighbors=5, p=1, weights=distance; total time=   6.7s
[CV] END metric=euclidean, n_neighbors=5, p=1, weights=uniform; total time=   6.8s
[CV] END metric=euclidean, n_neighbors=5, p=1, weights=uniform; total time=   6.8s
[CV] END metric=euclidean, n_neighbors=5, p=1, weights=uniform; total time=   6.8s
[CV] END metric=eucl

## 14E. Tune LinearSVC

In [27]:
print('\n' + '='*70)
print('TUNING: LinearSVC (RandomizedSearchCV on 50% sample)')
print('='*70)

model_name = 'LinearSVC'
start_time = time.time()

print('Creating 50% stratified subsample for faster tuning...\n')

# Create subsample
X_tune, _, y_tune, _ = train_test_split(
    X_train_scaled,
    y_train,
    train_size=0.5,
    stratify=y_train,
    random_state=42
)

print(f'Tuning on {len(y_tune):,} samples (50% of training data)')

# Define base LinearSVC (not wrapped yet)
# Parameter distributions - NO WRAPPER IN SEARCH
param_dist = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'max_iter': [3000, 5000],
    'tol': [1e-4, 1e-3]
}

base_linear_svc = LinearSVC(
    random_state=42,
    dual=False,
    loss='squared_hinge',
    class_weight='balanced'
)

# Search on UNWRAPPED LinearSVC
random_search = RandomizedSearchCV(
    base_linear_svc,  # Search on LinearSVC directly
    param_dist,
    n_iter=10,
    cv=5,
    scoring='f1',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_tune, y_tune)

print(f'\n✓ Tuning on subset complete')
print(f'  Best CV F1 (on subset): {random_search.best_score_:.4f}')
print(f'  Best parameters: {random_search.best_params_}')

# Extract best parameters and retrain on full data WITH CALIBRATION
print('\nRetraining best model on full training data with calibration...')
best_params = random_search.best_params_

final_linear_svc = LinearSVC(
    C=best_params['C'],
    max_iter=best_params['max_iter'],
    tol=best_params['tol'],
    random_state=42,
    dual=False,
    loss='squared_hinge',
    class_weight='balanced'
)

# NOW wrap with calibration
best_model = CalibratedClassifierCV(final_linear_svc, cv=3)
best_model.fit(X_train_scaled, y_train)

# Evaluate
y_train_pred = best_model.predict(X_train_scaled)
y_val_pred = best_model.predict(X_val_scaled)
y_test_pred = best_model.predict(X_test_scaled)

train_acc = accuracy_score(y_train, y_train_pred)
val_acc = accuracy_score(y_val, y_val_pred)
test_acc = accuracy_score(y_test, y_test_pred)
val_f1 = f1_score(y_val, y_val_pred)
test_f1 = f1_score(y_test, y_test_pred)

tune_time = time.time() - start_time

# Store
tuned_models[model_name] = best_model
tuning_results.append({
    'Model': model_name,
    'Train Acc': train_acc,
    'Val Acc': val_acc,
    'Test Acc': test_acc,
    'Val F1': val_f1,
    'Test F1': test_f1,
    'Tune Time (s)': tune_time
})

# Save immediately
model_path = MODELS_DIR / f'linearsvc_tuned_{FEATURE_SET}.pkl'
joblib.dump(best_model, model_path)

print(f'\n✓ {model_name} COMPLETE')
print(f'  Test F1:  {test_f1:.4f} (on full test set)')
print(f'  Test Acc: {test_acc:.4f}')
print(f'  Time:     {tune_time/60:.1f} minutes')
print(f'  Saved:    {model_path.name}')
print('='*70)



TUNING: LinearSVC (RandomizedSearchCV on 50% sample)
Creating 50% stratified subsample for faster tuning...

Tuning on 20,965 samples (50% of training data)
Fitting 5 folds for each of 10 candidates, totalling 50 fits

✓ Tuning on subset complete
  Best CV F1 (on subset): 0.9459
  Best parameters: {'tol': 0.001, 'max_iter': 3000, 'C': 100}

Retraining best model on full training data with calibration...

✓ LinearSVC COMPLETE
  Test F1:  0.9486 (on full test set)
  Test Acc: 0.9429
  Time:     0.9 minutes
  Saved:    linearsvc_tuned_combined.pkl


## 14F. Optimize GMM (BIC-based Model Selection)

In [28]:
print('\n' + '='*70)
print('OPTIMIZING: GMM (BIC-based component selection)')
print('='*70)

model_name = 'GMM'
start_time = time.time()

print('\nSearching for optimal number of components...\n')

n_components_range = [2, 4, 6, 8, 10, 12, 14]
covariance_types = ['diag', 'spherical']

best_bic = np.inf
best_gmm = None
best_n_components = None
best_cov_type = None

results_list = []

for cov_type in covariance_types:
    print(f'Testing covariance_type="{cov_type}":')
    for n_comp in n_components_range:
        gmm = GaussianMixture(
            n_components=n_comp,
            covariance_type=cov_type,
            random_state=42,
            max_iter=100
        )
        
        gmm.fit(X_train_scaled)
        bic = gmm.bic(X_val_scaled)
        log_likelihood = gmm.score(X_val_scaled)
        
        results_list.append({
            'n_components': n_comp,
            'covariance_type': cov_type,
            'BIC': bic,
            'Log-likelihood': log_likelihood
        })
        
        print(f'  n={n_comp:2d}: BIC={bic:12.2f}, log-likelihood={log_likelihood:8.2f}')
        
        if bic < best_bic:
            best_bic = bic
            best_gmm = gmm
            best_n_components = n_comp
            best_cov_type = cov_type
    print()

print(f'✓ Optimal configuration:')
print(f'  n_components: {best_n_components}')
print(f'  covariance_type: {best_cov_type}')
print(f'  BIC: {best_bic:.2f}')

# Map clusters to classes
print(f'\nMapping {best_n_components} clusters to classes...')
train_clusters = best_gmm.predict(X_train_scaled)

cluster_to_class = {}
for cluster_id in range(best_n_components):
    mask = train_clusters == cluster_id
    if mask.sum() > 0:
        cluster_labels = y_train[mask]
        majority_class = np.bincount(cluster_labels).argmax()
        cluster_to_class[cluster_id] = majority_class
        
        n_grinder = (cluster_labels == 1).sum()
        n_non_grinder = (cluster_labels == 0).sum()
        total = len(cluster_labels)
        purity = max(n_grinder, n_non_grinder) / total
        
        print(f'  Cluster {cluster_id} → {"Grinder" if majority_class == 1 else "Non-grinder"} '
              f'(purity: {purity:.1%}, n={total})')

# Define prediction function
def gmm_predict_optimized(X):
    clusters = best_gmm.predict(X)
    return np.array([cluster_to_class[c] for c in clusters])

# Evaluate
y_train_pred = gmm_predict_optimized(X_train_scaled)
y_val_pred = gmm_predict_optimized(X_val_scaled)
y_test_pred = gmm_predict_optimized(X_test_scaled)

train_acc = accuracy_score(y_train, y_train_pred)
val_acc = accuracy_score(y_val, y_val_pred)
test_acc = accuracy_score(y_test, y_test_pred)
val_f1 = f1_score(y_val, y_val_pred)
test_f1 = f1_score(y_test, y_test_pred)

tune_time = time.time() - start_time

# Store
tuning_results.append({
    'Model': model_name,
    'Train Acc': train_acc,
    'Val Acc': val_acc,
    'Test Acc': test_acc,
    'Val F1': val_f1,
    'Test F1': test_f1,
    'Tune Time (s)': tune_time
})

# Save GMM and mapping
model_path = MODELS_DIR / f'gmm_optimized_{FEATURE_SET}.pkl'
gmm_data = {
    'model': best_gmm,
    'cluster_to_class': cluster_to_class,
    'n_components': best_n_components,
    'covariance_type': best_cov_type
}
joblib.dump(gmm_data, model_path)

print(f'\n✓ {model_name} COMPLETE')
print(f'  Test F1:  {test_f1:.4f}')
print(f'  Test Acc: {test_acc:.4f}')
print(f'  Time:     {tune_time/60:.1f} minutes')
print(f'  Saved:    {model_path.name}')
print('='*70)



OPTIMIZING: GMM (BIC-based component selection)

Searching for optimal number of components...

Testing covariance_type="diag":
  n= 2: BIC=  5552519.39, log-likelihood= -309.45
  n= 4: BIC=  4853167.53, log-likelihood= -269.91
  n= 6: BIC=  4580268.33, log-likelihood= -254.17
  n= 8: BIC=  4369633.26, log-likelihood= -241.90
  n=10: BIC=  4259572.76, log-likelihood= -235.25
  n=12: BIC=  4195483.54, log-likelihood= -231.17
  n=14: BIC=  4122495.93, log-likelihood= -226.59

Testing covariance_type="spherical":
  n= 2: BIC=  6038205.72, log-likelihood= -336.81
  n= 4: BIC=  5339860.70, log-likelihood= -297.58
  n= 6: BIC=  5176460.27, log-likelihood= -288.20
  n= 8: BIC=  5054599.78, log-likelihood= -281.14
  n=10: BIC=  4976528.43, log-likelihood= -276.53
  n=12: BIC=  4927898.71, log-likelihood= -273.56
  n=14: BIC=  4877466.63, log-likelihood= -270.49

✓ Optimal configuration:
  n_components: 14
  covariance_type: diag
  BIC: 4122495.93

Mapping 14 clusters to classes...
  Cluster 0

---
# ANALYSIS & COMPARISON

---

## 15. Compare Baseline vs Tuned

In [29]:
print('\n' + '='*70)
print('COMPARING: Baseline vs Tuned Models')
print('='*70)

# Create tuning DataFrame
df_tuning = pd.DataFrame(tuning_results)

# Save tuning results
tuning_csv = RESULTS_DIR / f'05_tuning_results_{FEATURE_SET}.csv'
df_tuning.to_csv(tuning_csv, index=False)
print(f'✓ Saved tuning results: {tuning_csv.name}\n')

# Create comparison
comparison_data = []

for _, tuned_row in df_tuning.iterrows():
    model_name = tuned_row['Model']
    
    # Find baseline (handle GMM separately)
    baseline_rows = df_baseline[df_baseline['Model'] == model_name]
    
    if len(baseline_rows) == 0:
        print(f'⚠️  No baseline found for {model_name}')
        continue
    
    baseline_row = baseline_rows.iloc[0]
    
    comparison_data.append({
        'Model': model_name,
        'Baseline Test Acc': baseline_row['Test Acc'],
        'Tuned Test Acc': tuned_row['Test Acc'],
        'Acc Improvement': tuned_row['Test Acc'] - baseline_row['Test Acc'],
        'Baseline Test F1': baseline_row['Test F1'],
        'Tuned Test F1': tuned_row['Test F1'],
        'F1 Improvement': tuned_row['Test F1'] - baseline_row['Test F1'],
        'Train-Test Gap': tuned_row['Train Acc'] - tuned_row['Test Acc']
    })

df_comparison = pd.DataFrame(comparison_data)

# Display comparison
print('BASELINE vs TUNED COMPARISON:')
print(df_comparison.to_string(index=False))

# Check for overfitting
print('\nOVERFITTING CHECK:')
for _, row in df_comparison.iterrows():
    gap = row['Train-Test Gap']
    if gap > 0.05:
        print(f'  ⚠️  {row["Model"]}: Overfitting detected (gap={gap:.3f})')
    elif gap < -0.02:
        print(f'  ⚠️  {row["Model"]}: Underfitting (gap={gap:.3f})')
    else:
        print(f'  ✓ {row["Model"]}: Good fit (gap={gap:.3f})')

# Save comparison
comparison_csv = RESULTS_DIR / f'05_comparison_summary_{FEATURE_SET}.csv'
df_comparison.to_csv(comparison_csv, index=False)
print(f'\n✓ Saved comparison: {comparison_csv.name}')
print('='*70)


COMPARING: Baseline vs Tuned Models
✓ Saved tuning results: 05_tuning_results_combined.csv

BASELINE vs TUNED COMPARISON:
        Model  Baseline Test Acc  Tuned Test Acc  Acc Improvement  Baseline Test F1  Tuned Test F1  F1 Improvement  Train-Test Gap
      XGBoost           0.985302        0.987418         0.002116          0.986683       0.988594        0.001911        0.012558
Random Forest           0.977397        0.977063        -0.000334          0.979625       0.979326       -0.000299        0.016236
          KNN           0.970827        0.985970         0.015143          0.973726       0.987301        0.013575        0.014006
    LinearSVC           0.941877        0.942879         0.001002          0.947590       0.948582        0.000991        0.001839
          GMM           0.692351        0.778755         0.086405          0.747510       0.810491        0.062981        0.001527

OVERFITTING CHECK:
  ✓ XGBoost: Good fit (gap=0.013)
  ✓ Random Forest: Good fit (gap=0.01

## 16. Identify Best Model

In [30]:
print('\n' + '='*70)
print('BEST MODEL IDENTIFICATION')
print('='*70)

# Find best by test F1
best_idx = df_comparison['Tuned Test F1'].idxmax()
best_row = df_comparison.iloc[best_idx]
best_model_name = best_row['Model']
best_model = tuned_models.get(best_model_name)

print(f'\nBest Model: {best_model_name}')
print(f'  Test F1:        {best_row["Tuned Test F1"]:.4f}')
print(f'  Test Accuracy:  {best_row["Tuned Test Acc"]:.4f}')
print(f'  F1 Improvement: +{best_row["F1 Improvement"]:.4f} ({best_row["F1 Improvement"]/best_row["Baseline Test F1"]*100:.1f}%)')
print(f'  Train-Test Gap: {best_row["Train-Test Gap"]:.4f}')

# Detailed classification report
if best_model is not None:
    y_pred_best = best_model.predict(X_test_scaled)
    
    print('\nDETAILED CLASSIFICATION REPORT:')
    print(classification_report(y_test, y_pred_best, target_names=['Non-grinder', 'Grinder'], digits=4))
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred_best)
    print('\nCONFUSION MATRIX:')
    print(cm)
    print(f'\n  True Negatives:  {cm[0,0]:,} (correct non-grinder)')
    print(f'  False Positives: {cm[0,1]:,} (false alarms)')
    print(f'  False Negatives: {cm[1,0]:,} (missed grinders - CRITICAL!)')
    print(f'  True Positives:  {cm[1,1]:,} (correct grinder)')
    
    # Rates
    fpr = cm[0,1] / (cm[0,1] + cm[0,0])
    fnr = cm[1,0] / (cm[1,0] + cm[1,1])
    print(f'\n  False Positive Rate: {fpr:.2%} (false alarms)')
    print(f'  False Negative Rate: {fnr:.2%} (missed thefts)')

print('='*70)


BEST MODEL IDENTIFICATION

Best Model: XGBoost
  Test F1:        0.9886
  Test Accuracy:  0.9874
  F1 Improvement: +0.0019 (0.2%)
  Train-Test Gap: 0.0126

DETAILED CLASSIFICATION REPORT:
              precision    recall  f1-score   support

 Non-grinder     0.9945    0.9776    0.9860      4062
     Grinder     0.9818    0.9955    0.9886      4919

    accuracy                         0.9874      8981
   macro avg     0.9881    0.9866    0.9873      8981
weighted avg     0.9875    0.9874    0.9874      8981


CONFUSION MATRIX:
[[3971   91]
 [  22 4897]]

  True Negatives:  3,971 (correct non-grinder)
  False Positives: 91 (false alarms)
  False Negatives: 22 (missed grinders - CRITICAL!)
  True Positives:  4,897 (correct grinder)

  False Positive Rate: 2.24% (false alarms)
  False Negative Rate: 0.45% (missed thefts)


## 17. Visualize Results

In [31]:
print('\nGenerating visualizations...')

# Figure 1: Baseline vs Tuned Comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

models_list = df_comparison['Model'].tolist()
x = range(len(models_list))
width = 0.35

# Accuracy comparison
axes[0].bar([i - width/2 for i in x], df_comparison['Baseline Test Acc'], width, label='Baseline', alpha=0.8, color='skyblue')
axes[0].bar([i + width/2 for i in x], df_comparison['Tuned Test Acc'], width, label='Tuned', alpha=0.8, color='orange')
axes[0].set_xlabel('Model', fontsize=12)
axes[0].set_ylabel('Test Accuracy', fontsize=12)
axes[0].set_title('Baseline vs Tuned - Test Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models_list, rotation=45, ha='right')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_ylim([0.8, 1.0])

# F1 comparison
axes[1].bar([i - width/2 for i in x], df_comparison['Baseline Test F1'], width, label='Baseline', alpha=0.8, color='skyblue')
axes[1].bar([i + width/2 for i in x], df_comparison['Tuned Test F1'], width, label='Tuned', alpha=0.8, color='orange')
axes[1].set_xlabel('Model', fontsize=12)
axes[1].set_ylabel('Test F1 Score', fontsize=12)
axes[1].set_title('Baseline vs Tuned - Test F1 Score', fontsize=14, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(models_list, rotation=45, ha='right')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)
axes[1].set_ylim([0.8, 1.0])

plt.tight_layout()
comparison_fig = FIGURES_DIR / f'05_model_comparison_{FEATURE_SET}.png'
plt.savefig(comparison_fig, dpi=300, bbox_inches='tight')
print(f'✓ Saved: {comparison_fig.name}')
plt.close()

# Figure 2: Confusion Matrix for Best Model
if best_model is not None:
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, 
                xticklabels=['Non-grinder', 'Grinder'],
                yticklabels=['Non-grinder', 'Grinder'])
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('Actual', fontsize=12)
    ax.set_title(f'Confusion Matrix - {best_model_name} ({FEATURE_SET.upper()})', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    cm_fig = FIGURES_DIR / f'05_confusion_matrix_best_{FEATURE_SET}.png'
    plt.savefig(cm_fig, dpi=300, bbox_inches='tight')
    print(f'✓ Saved: {cm_fig.name}')
    plt.close()

# Figure 3: Feature Importance (if tree-based model)
if best_model_name in ['XGBoost', 'Random Forest', 'Gradient Boosting'] and hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    indices = np.argsort(importances)[::-1][:30]  # Top 30
    
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(range(len(indices)), importances[indices], color='steelblue')
    ax.set_yticks(range(len(indices)))
    ax.set_yticklabels([f'Feature {i}' for i in indices])
    ax.set_xlabel('Importance', fontsize=12)
    ax.set_ylabel('Feature', fontsize=12)
    ax.set_title(f'Top 30 Feature Importances - {best_model_name} ({FEATURE_SET.upper()})', fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    
    plt.tight_layout()
    fi_fig = FIGURES_DIR / f'05_feature_importance_best_{FEATURE_SET}.png'
    plt.savefig(fi_fig, dpi=300, bbox_inches='tight')
    print(f'✓ Saved: {fi_fig.name}')
    plt.close()

print('\n✓ All visualizations complete')


Generating visualizations...
✓ Saved: 05_model_comparison_combined.png
✓ Saved: 05_confusion_matrix_best_combined.png
✓ Saved: 05_feature_importance_best_combined.png

✓ All visualizations complete


## 18. Production Readiness Check

In [32]:
print('\n' + '='*70)
print('PRODUCTION READINESS CHECK')
print('='*70)

if best_model is not None:
    # Model size
    import pickle
    import sys
    
    model_path = MODELS_DIR / f'{best_model_name.lower().replace(" ", "_")}_tuned_{FEATURE_SET}.pkl'
    if model_path.exists():
        model_size_mb = model_path.stat().st_size / 1e6
    else:
        model_size_mb = sys.getsizeof(pickle.dumps(best_model)) / 1e6
    
    # Inference time
    import time
    n_samples = 100
    start = time.time()
    _ = best_model.predict(X_test_scaled[:n_samples])
    inference_time_ms = (time.time() - start) / n_samples * 1000
    
    # Memory requirement (rough estimate)
    memory_mb = sys.getsizeof(pickle.dumps(best_model)) / 1e6
    
    print(f'\nBest Model: {best_model_name}')
    print(f'  Model size:     {model_size_mb:.1f} MB')
    print(f'  Inference time: {inference_time_ms:.2f} ms/sample')
    print(f'  Memory usage:   ~{memory_mb:.1f} MB RAM')
    
    # Thresholds
    size_ok = model_size_mb < 100
    speed_ok = inference_time_ms < 50
    memory_ok = memory_mb < 500
    
    print(f'\nProduction Thresholds:')
    print(f'  Size < 100 MB:     {"✓" if size_ok else "⚠️ "} ({model_size_mb:.1f} MB)')
    print(f'  Speed < 50 ms:     {"✓" if speed_ok else "⚠️ "} ({inference_time_ms:.2f} ms)')
    print(f'  Memory < 500 MB:   {"✓" if memory_ok else "⚠️ "} ({memory_mb:.1f} MB)')
    
    prod_ready = size_ok and speed_ok and memory_ok
    
    print(f'\nOverall Status: {"✓ PRODUCTION READY" if prod_ready else "⚠️  NEEDS OPTIMIZATION"}')
    
    if not prod_ready:
        print('\nRecommendations:')
        if not size_ok:
            print('  - Consider model compression or pruning')
        if not speed_ok:
            print('  - Consider simpler model or model optimization')
        if not memory_ok:
            print('  - Consider model quantization')

print('='*70)


PRODUCTION READINESS CHECK

Best Model: XGBoost
  Model size:     1.3 MB
  Inference time: 0.02 ms/sample
  Memory usage:   ~1.3 MB RAM

Production Thresholds:
  Size < 100 MB:     ✓ (1.3 MB)
  Speed < 50 ms:     ✓ (0.02 ms)
  Memory < 500 MB:   ✓ (1.3 MB)

Overall Status: ✓ PRODUCTION READY


## 19. Log to MLflow

In [33]:
print('\n' + '='*70)
print('LOGGING TO MLFLOW')
print('='*70)

with mlflow.start_run(run_name=f'05_classical_training_{FEATURE_SET}'):
    
    # Parameters
    mlflow.log_param('notebook', '05')
    mlflow.log_param('stage', 'classical_training')
    mlflow.log_param('feature_set', FEATURE_SET)
    mlflow.log_param('num_features', X.shape[1])
    mlflow.log_param('train_samples_original', len(y_train))
    mlflow.log_param('train_samples_balanced', len(y_train))
    mlflow.log_param('smote_strategy', 'undersample_then_smote')
    mlflow.log_param('target_per_class', TARGET_PER_CLASS)
    mlflow.log_param('synthetic_percentage', synthetic_percentage)
    mlflow.log_param('cv_folds', 5)
    mlflow.log_param('models_trained', len(baseline_models) + 1)  # +1 for GMM
    mlflow.log_param('models_tuned', len(tuned_models))
    
    # Baseline metrics
    for _, row in df_baseline.iterrows():
        model_name = row['Model'].replace(' ', '_').lower()
        mlflow.log_metric(f'{model_name}_baseline_test_acc', row['Test Acc'])
        mlflow.log_metric(f'{model_name}_baseline_test_f1', row['Test F1'])
        mlflow.log_metric(f'{model_name}_baseline_train_time_s', row['Train Time (s)'])
    
    # Tuned metrics
    for _, row in df_tuning.iterrows():
        model_name = row['Model'].replace(' ', '_').lower()
        mlflow.log_metric(f'{model_name}_tuned_test_acc', row['Test Acc'])
        mlflow.log_metric(f'{model_name}_tuned_test_f1', row['Test F1'])
        mlflow.log_metric(f'{model_name}_tune_time_s', row['Tune Time (s)'])
    
    # Comparison metrics
    for _, row in df_comparison.iterrows():
        model_name = row['Model'].replace(' ', '_').lower()
        mlflow.log_metric(f'{model_name}_f1_improvement', row['F1 Improvement'])
        mlflow.log_metric(f'{model_name}_overfitting_gap', row['Train-Test Gap'])
    
    # Best model metrics
    mlflow.log_metric('best_model_test_f1', best_row['Tuned Test F1'])
    mlflow.log_metric('best_model_test_acc', best_row['Tuned Test Acc'])
    if best_model is not None:
        mlflow.log_metric('best_model_inference_ms', inference_time_ms)
        mlflow.log_metric('best_model_size_mb', model_size_mb)
    
    # Tags
    mlflow.set_tags({
        'best_model': best_model_name,
        'feature_set': FEATURE_SET,
        'production_ready': 'yes' if prod_ready else 'no',
        'balancing_method': 'minimal_smote',
        'tuning_methods': 'optuna_xgb_halving_rf_sequential_gb',
        'skipped_models': 'SVM_RBF'
    })
    
    # Artifacts
    mlflow.log_artifact(str(baseline_csv))
    mlflow.log_artifact(str(tuning_csv))
    mlflow.log_artifact(str(comparison_csv))
    mlflow.log_artifact(str(comparison_fig))
    if cm_fig.exists():
        mlflow.log_artifact(str(cm_fig))
    
    # Best model
    if best_model is not None:
        mlflow.sklearn.log_model(best_model, f'best_model_{FEATURE_SET}')
    
    print('\n✓ All data logged to MLflow')
    print(f'  Run ID: {mlflow.active_run().info.run_id}')

print('='*70)


LOGGING TO MLFLOW


NameError: name 'TARGET_PER_CLASS' is not defined

## 20. Summary

In [ ]:
notebook_total_time = time.time() - notebook_start_time

print('\n' + '='*70)
print('CLASSICAL TRAINING & TUNING COMPLETE')
print('='*70)

print(f'\nFeature Set: {FEATURE_SET.upper()}')
print(f'Total Features: {X.shape[1]}')
print(f'Training Samples: {len(y_train):,} ({synthetic_percentage:.1f}% synthetic)')

print(f'\nModels Trained (Baseline): {len(baseline_models) + 1}')
for name in list(baseline_models.keys()) + ['GMM']:
    print(f'  ✓ {name}')

print(f'\nModels Tuned: {len(tuned_models)}')
for name, improvement in zip(df_comparison['Model'], df_comparison['F1 Improvement']):
    print(f'  ✓ {name} (+{improvement:.4f} F1, {improvement/df_comparison[df_comparison["Model"]==name]["Baseline Test F1"].values[0]*100:+.1f}%)')

print(f'\nBest Model: {best_model_name}')
print(f'  Test F1:       {best_row["Tuned Test F1"]:.4f}')
print(f'  Test Accuracy: {best_row["Tuned Test Acc"]:.4f}')
print(f'  Improvement:   +{best_row["F1 Improvement"]:.4f} ({best_row["F1 Improvement"]/best_row["Baseline Test F1"]*100:.1f}%)')
print(f'  Overfitting:   {best_row["Train-Test Gap"]:.4f}')

print(f'\nTiming:')
print(f'  Baseline training: {baseline_total_time/60:.1f} min')
print(f'  Hyperparameter tuning: {df_tuning["Tune Time (s)"].sum()/60:.1f} min')
print(f'  Total notebook time: {notebook_total_time/60:.1f} min')

if best_model is not None:
    print(f'\nProduction Readiness:')
    print(f'  Model size:    {model_size_mb:.1f} MB')
    print(f'  Inference:     {inference_time_ms:.2f} ms/sample')
    print(f'  Status:        {"✓ READY" if prod_ready else "⚠️  NEEDS OPTIMIZATION"}')

print(f'\nSaved Artifacts:')
print(f'  Models:        {len(tuned_models)} tuned models in {MODELS_DIR.name}/')
print(f'  Results:       3 CSV files in {RESULTS_DIR.name}/')
print(f'  Figures:       {len(list(FIGURES_DIR.glob(f"05_*_{FEATURE_SET}.png")))} figures in {FIGURES_DIR.name}/')
print(f'  MLflow:        Run logged with ID {mlflow.active_run().info.run_id if mlflow.active_run() else "N/A"}')

print(f'\nNext Steps:')
print(f'  1. Run with other feature sets (change FEATURE_SET in cell 4):')
print(f'     - gtcc: GTCC features')
print(f'     - combined: MFCC + GTCC combined')
print(f'  2. Compare feature sets using results CSVs')
print(f'  3. Proceed to:')
print(f'     → 06a_classical_optimization.ipynb (ensembles, stacking)')
print(f'     → 06b_feature_importance_expanded.ipynb (SHAP analysis)')
print(f'     → 07_custom_cnn_training.ipynb (deep learning)')

print('\n' + '='*70)
print('✓ NOTEBOOK 05 COMPLETE')
print('='*70)


CLASSICAL TRAINING & TUNING COMPLETE

Feature Set: COMBINED
Total Features: 260
Training Samples: 42,266 (4.6% synthetic)

Models Trained (Baseline): 6
  ✓ LinearSVC
  ✓ Random Forest
  ✓ XGBoost
  ✓ Gradient Boosting
  ✓ KNN
  ✓ GMM

Models Tuned: 4
  ✓ XGBoost (+-0.0002 F1, -0.0%)
  ✓ Random Forest (+0.0004 F1, +0.0%)
  ✓ KNN (+0.0209 F1, +2.2%)
  ✓ LinearSVC (+0.0008 F1, +0.1%)
  ✓ GMM (+0.0159 F1, +2.2%)

Best Model: XGBoost
  Test F1:       0.9877
  Test Accuracy: 0.9864
  Improvement:   +-0.0002 (-0.0%)
  Overfitting:   0.0132

Timing:
  Baseline training: 11.1 min
  Hyperparameter tuning: 34.8 min
  Total notebook time: 46.0 min

Production Readiness:
  Model size:    1.7 MB
  Inference:     0.01 ms/sample
  Status:        ✓ READY

Saved Artifacts:
  Models:        4 tuned models in classical/
  Results:       3 CSV files in results/
  Figures:       3 figures in figures/
  MLflow:        Run logged with ID N/A

Next Steps:
  1. Run with other feature sets (change FEATURE_SET i